In [5]:
from dotenv import load_dotenv
load_dotenv()

True

## 定义模型

In [6]:
from langchain.chat_models import init_chat_model
import os

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    base_url = os.getenv("DASHSCOPE_BASE_URL"),
    api_key = os.getenv("DASHSCOPE_API_KEY")
)

## 定义工具

In [7]:
from langchain_tavily import TavilySearch

# web搜索工具，使用tavily作为web搜索工具
web_search = TavilySearch(
    max_result = 5,
    topic = "general"
)

## 添加记忆管理

In [8]:
# 采用sqlite
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# 连接sqlite
connection = sqlite3.connect("resources/personal_chief.db", check_same_thread=False)

# 初始化checkpointer
checkpointer = SqliteSaver(connection)

# 自动建表
checkpointer.setup()

## 定义智能体

In [9]:
from langchain.agents import create_agent
system_prompt = """
你是一名私人厨师。收到用户提供的食材照片或清单后，请按以下流程操作[reference:6][reference:7]：

1.  **识别和评估食材**：若用户提供照片，首先辨识所有可见食材[reference:8]。基于食材的外观状态，评估其新鲜度与可用量，整理出一份“当前可用食材清单”[reference:9][reference:10]。
2.  **智能食谱检索**：**优先调用 `web_search` 工具**，以“可用食材清单”为核心关键词，查找可行菜谱[reference:11][reference:12]。
3.  **多维度评估与排序**：从营养价值和制作难度两个维度对检索到的候选食谱进行量化打分，并根据得分排序，让制作简单且营养丰富的食谱排名靠前[reference:13][reference:14]。
4.  **结构化方案输出**：把排序后的食谱整理为一份结构清晰的建议报告，要包含食谱信息、得分、推荐理由、食谱的参考图片，帮助用户快速做出决策[reference:15][reference:16]。

请**严格按照流程**，**优先调用 `web_search` 工具**搜索食谱，搜索不到的情况下才能自己发挥[reference:17][reference:18]。
"""

agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=checkpointer
)

## 测试

In [ ]:
from langchain.messages import HumanMessage

multimodal_message = HumanMessage([
    {"type": "text", "text": "帮我看看能做什么。"},
    {"type": "image_url", "url": "https://pic.nximg.cn/file/20230202/33857552_195013311106_2.jpg"}
])

config = {"configurable": {"thread_id": "1"}}

In [11]:
response = agent.invoke({"messages": [multimodal_message]}, config)

In [12]:
# 输出
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么。'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230202/33857552_195013311106_2.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_9beeb6291a784ccb9a160c64)
 Call ID: call_9beeb6291a784ccb9a160c64
  Args:
    query: 芦笋 蟹味菇 胡萝卜 食谱 做法
    search_depth: basic
  tavily_search (call_0a93c59200c540b992aba682)
 Call ID: call_0a93c59200c540b992aba682
  Args:
    query: 茄子 四季豆 大蒜 食谱 做法
    search_depth: basic
  tavily_search (call_e22a9fa376f24b1ea121c25a)
 Call ID: call_e22a9fa376f24b1ea121c25a
  Args:
    query: 玉米笋 黄瓜 圣女果 沙拉 食谱
    search_depth: basic
  tavily_search (call_db287d282a04450498947ff7)
 Call ID: call_db287d282a04450498947ff7
  Args:
    query: 青豆 胡萝卜 玉米笋 炒菜 食谱
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search



In [14]:
response = agent.invoke(
    {"messages": [HumanMessage(content="我喜欢第3道菜，可以说详细点吗？")]},
    config
)

In [15]:
for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

[{'type': 'text', 'text': '帮我看看能做什么。'}, {'type': 'image', 'url': 'https://pic.nximg.cn/file/20230202/33857552_195013311106_2.jpg'}]
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_9beeb6291a784ccb9a160c64)
 Call ID: call_9beeb6291a784ccb9a160c64
  Args:
    query: 芦笋 蟹味菇 胡萝卜 食谱 做法
    search_depth: basic
  tavily_search (call_0a93c59200c540b992aba682)
 Call ID: call_0a93c59200c540b992aba682
  Args:
    query: 茄子 四季豆 大蒜 食谱 做法
    search_depth: basic
  tavily_search (call_e22a9fa376f24b1ea121c25a)
 Call ID: call_e22a9fa376f24b1ea121c25a
  Args:
    query: 玉米笋 黄瓜 圣女果 沙拉 食谱
    search_depth: basic
  tavily_search (call_db287d282a04450498947ff7)
 Call ID: call_db287d282a04450498947ff7
  Args:
    query: 青豆 胡萝卜 玉米笋 炒菜 食谱
    search_depth: basic
================================= Tool Message =================================
Name: tavily_search



In [16]:
response["messages"][-1].pretty_print()

================================== Ai Message ==================================

没问题！这道**「青豆玉米笋炒胡萝卜」**不仅颜色红绿黄相间、让人食欲大开，而且口感层次非常丰富：青豆绵软清甜、玉米笋脆嫩爽口、胡萝卜带着自然的甘甜。

作为您的私人厨师，我为您整理了这道菜的**详细进阶版做法**，保证您在家也能做出餐厅级别的清爽小炒！

---

### 🥗 青豆玉米笋炒胡萝卜（详细版）

#### 🛒 食材清单（2人份）
*   **主料**：新鲜青豆 100g（或冷冻青豆）、玉米笋 150g、胡萝卜 半根（约100g）
*   **配料**：大蒜 3-4瓣
*   **调料**：食用油 2汤匙、盐 适量、鸡精或干贝粉 少许、水淀粉 1汤匙（可选，用于勾薄芡）

---

#### 🔪 详细烹饪步骤

**第一步：食材预处理**
1.  **玉米笋**：洗净后，如果比较粗，可以斜刀切成段，或者对半切开；如果比较细嫩，整根保留即可。
2.  **胡萝卜**：去皮，切成菱形片或薄片（切薄一点更容易熟，且摆盘好看）。
3.  **大蒜**：切成蒜末备用。
4.  **青豆**：如果是新鲜青豆，剥壳洗净；如果是冷冻青豆，提前拿出来室温解冻。

**第二步：蔬菜焯水（关键步骤，保持颜色翠绿）**
1.  锅中烧开水，水开后加入**一小勺盐和几滴食用油**（这是让蔬菜保持鲜亮翠绿的秘诀）。
2.  先放入**胡萝卜片**焯水约 1 分钟。
3.  接着放入**玉米笋和青豆**，继续焯水 1-2 分钟，直到青豆断生、颜色变深绿。
4.  将蔬菜全部捞出，**迅速放入冰水或凉开水中过凉**（这一步能保持蔬菜脆嫩的口感），然后彻底沥干水分备用。

**第三步：爆香与翻炒**
1.  炒锅烧热，倒入食用油，油温五成热时，放入**蒜末**小火爆香，炒出浓郁的蒜香味（注意不要炒焦）。
2.  转**大火**，倒入沥干水分的青豆、玉米笋和胡萝卜片，快速翻炒均匀，让食材裹上蒜油。

**第四步：调味与出锅**
1.  加入适量的**盐**和少许**鸡精**（如果有干贝粉或松茸鲜，加一点提鲜效果更佳）。
2.  翻炒均匀后，如果觉得锅里太干，可以沿着锅边淋入**一小勺热水**，让味道融合。
3.  **（可选）*

## 流式调用
有工具调用的agent消息类型比较多，要正确的流式输出比较麻烦

In [ ]:
from langchain.messages import AIMessageChunk

for chunk, metadata in agent.stream(
    {"messages": [multimodal_message]},
    config,
    stream_mode="messages"
):
    # 在生成用户想要回答前，会有很多不重要消息或空消息
    if isinstance(chunk, AIMessageChunk) and chunk.content:
        print(chunk.content, end="", flush=True)

## 获取会话历史

In [ ]:
checkpointer.get(config)

In [ ]:
for m in checkpointer.get(config)['channel_values']['messages']:
    print(type(m))
    print(m)

定义一个便捷的获取会话历史的函数

In [ ]:
from langchain.messages import AIMessage

def get_messages(thread_id: str) -> list[dict[str, str]]:
    """获取会话历史"""

    # 根据 thread_id 查询 checkpoint
    cp = checkpointer.get({"configurable": {"thread_id": thread_id}})

    # 如果不存在，返回空列表
    if not cp:
        return []

    # 安全获取 messages
    channel_values = cp.get("channel_values")
    if not channel_values:
        return []

    messages = channel_values.get("messages", [])
    if not messages:
        return []

    # 转换消息格式
    result = []
    for msg in message:
        if not msg.content:
            continue
        if isinstance(msg, HumanMessage):
            result.append({"rule": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            result.append({"rule": "assistant", "content": msg.content})

    return result

## 清空会话历史

In [ ]:
checkpointer.delete_thread("6")